# Kronos PoC: Kronos-small vs GBM on Japanese Stocks

Spec: `docs/superpowers/specs/2026-05-02-kronos-poc-design.md`. Run all cells top to bottom. See `notebooks/README.md` for details.

## 1. Setup

In [ ]:
# Cell 1: setup
!pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 tqdm==4.67.1
# torch / numpy / pandas / matplotlib are pre-installed on Colab; do not pin to avoid breaking Colab base env.

import os, json, time, hashlib, pickle, getpass
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from tqdm.auto import tqdm

print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}, mps={torch.backends.mps.is_available()}")

## 2. Auth

In [ ]:
# Cell 2: auth
# Try Colab Secrets first, fall back to interactive getpass. The key never gets printed.
try:
    from google.colab import userdata
    JQUANTS_API_KEY = userdata.get("JQUANTS_API_KEY")
    print("Loaded JQUANTS_API_KEY from Colab Secrets.")
except Exception:
    JQUANTS_API_KEY = None

if not JQUANTS_API_KEY:
    JQUANTS_API_KEY = getpass.getpass("JQuants API key (Premium plan): ").strip()

assert JQUANTS_API_KEY, "API key required"
os.environ["JQUANTS_API_KEY"] = JQUANTS_API_KEY
print(f"API key loaded ({len(JQUANTS_API_KEY)} chars). Header will be sent as 'x-api-key: ****'.")